In [1]:
!pip install pandas numpy scipy scikit-learn tensorflow matplotlib

In [6]:
!wget https://physionet.org/static/published-projects/gaitpdb/gait-in-parkinsons-disease-1.0.0.zip -O gaitpdb.zip
!unzip -q gaitpdb.zip -d gaitpdb_data

--2026-08-23 07:16:27--  https://physionet.org/static/published-projects/gaitpdb/gait-in-parkinsons-disease-1.0.0.zip
Resolving physionet.org (physionet.org)... 18.25.8.254
Connecting to physionet.org (physionet.org)|18.25.8.254|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://physionet.org/content/gaitpdb/get-zip/1.0.0/ [following]
--2026-08-23 07:16:28--  https://physionet.org/content/gaitpdb/get-zip/1.0.0/
Reusing existing connection to physionet.org:443.
HTTP request sent, awaiting response... 200 OK
Length: 302436928 (288M) [application/zip]
Saving to: ‘gaitpdb.zip’

gaitpdb.zip         100%[===================>] 288.43M   256KB/s    in 19m 41s 

2026-08-23 07:36:08 (250 KB/s) - ‘gaitpdb.zip’ saved [302436928/302436928]



In [8]:
import os
print(os.listdir('gaitpdb_data'))

['gait-in-parkinsons-disease-1.0.0']


In [12]:
import os
print(os.listdir('gaitpdb_data/gait-in-parkinsons-disease-1.0.0'))

['JuCo19_01.txt', 'JuPt02_01.txt', 'GaPt20_01.txt', 'JuPt10_01.txt', 'GaCo04_02.txt', 'GaPt17_10.txt', 'demographics.txt', 'GaPt33_01.txt', 'SiPt07_01.txt', 'SiCo24_01.txt', 'GaPt26_10.txt', 'GaPt29_02.txt', 'SiPt36_01.txt', 'GaCo17_10.txt', 'JuPt28_03.txt', 'GaPt13_02.txt', 'SiCo14_01.txt', 'JuCo26_01.txt', 'JuPt03_07.txt', 'JuPt15_04.txt', 'GaCo14_02.txt', 'GaCo16_02.txt', 'GaPt16_02.txt', 'SiCo27_01.txt', 'GaCo22_01.txt', 'GaPt23_10.txt', 'JuPt15_02.txt', 'JuCo17_01.txt', 'GaPt28_01.txt', 'GaPt23_01.txt', 'JuPt28_01.txt', 'JuPt03_02.txt', 'GaPt12_02.txt', 'SiCo05_01.txt', 'JuPt10_03.txt', 'SiPt29_01.txt', 'GaPt15_01.txt', 'SiPt39_01.txt', 'JuCo07_01.txt', 'JuPt11_03.txt', 'JuPt26_07.txt', 'JuPt28_07.txt', 'GaPt14_01.txt', 'JuPt06_03.txt', 'JuPt29_06.txt', 'GaPt24_02.txt', 'GaCo01_01.txt', 'GaPt12_01.txt', 'SiCo26_01.txt', 'JuPt11_04.txt', 'GaPt09_01.txt', 'SiCo03_01.txt', 'JuCo09_01.txt', 'JuPt10_06.txt', 'JuPt10_04.txt', 'GaPt18_10.txt', 'SiPt28_01.txt', 'JuCo05_01.txt', 'JuPt03_05

In [13]:
df = pd.read_csv('gaitpdb_data/gait-in-parkinsons-disease-1.0.0/GaCo04_02.txt', sep='\t', header=None)
print(df.shape)
df.head()

(12119, 19)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18
0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,59.51,222.64,178.42,65.45,85.25,20.13,24.20,19.47,0.0,675.07
1,0.01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,59.51,222.64,178.42,65.45,85.25,20.13,24.20,19.47,0.0,675.07
2,0.02,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,59.51,224.62,173.69,67.98,80.30,17.60,22.11,19.47,0.0,665.28
3,0.03,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,57.20,222.64,169.07,67.98,75.24,20.13,22.11,19.47,0.0,653.84
4,0.04,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,54.89,222.64,162.14,70.40,72.71,17.60,20.13,19.47,0.0,639.98


In [16]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copytree('gaitpdb_data', '/content/drive/MyDrive/gait_project/gaitpdb_data')

Mounted at /content/drive


'/content/drive/MyDrive/gait_project/gaitpdb_data'

In [1]:
# Removing the sensor noise

from scipy.signal import butter, filtfilt

def lowpass_filter(signal, cutoff=10, fs=100, order=4):
    nyquist = 0.5 * fs
    normal_cutoff = cutoff / nyquist
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    return filtfilt(b, a, signal)

In [2]:
# Orientation correction (for phone-based accelerometer data)

import numpy as np

def acceleration_magnitude(ax, ay, az):
    return np.sqrt(ax**2 + ay**2 + az**2)

In [4]:
# Gait cycle segmentation (peak detection):

from scipy.signal import find_peaks

def segment_gait_cycles(signal, fs=100, min_step_interval=0.4):
    distance = int(min_step_interval * fs)
    peaks, _ = find_peaks(signal, distance=distance, prominence=0.1)
    cycles = [signal[peaks[i]:peaks[i+1]] for i in range(len(peaks)-1)]
    return peaks, cycles